# Example 4: temporal displacement metrics on a speckle scan

**Goal.** Load an experimental speckle stack, apply flat-field correction, track a representative speckle motif between frames, and compare the measured trajectory against the nominal scan path.

This example combines two complementary levels of the API:

- `dip.signal.track_translation(...)` for **explicit motif tracking** between two frames.

In [ ]:
__author__ = ['Rafael Celestre']
__contact__ = 'rafael.celestre@esrf.eu'
__license__ = 'CECILL-2.1'
__copyright__ = 'ESRF - The European Synchrotron'
__created__ = '2026.03.25'
__changed__ = '2026.03.25'

import logging
import sys

import barc4dip as dip
import numpy as np

logging.basicConfig(level=logging.INFO, format="%(message)s")

print(sys.executable)
print(sys.version)

%load_ext autoreload
%autoreload 2

# %matplotlib widget
# %matplotlib inline

## 1) Load a speckle scan

Replace `data_path` with the location of the example dataset.  
Later this can point to the reproducible dataset archived in Zenodo.

In [ ]:
data_path = "/Users/celestre/work/experimental_data/speckle_tracking/speckle_tracking/speckles.h5"

scan = dip.read_image(data_path, verbose=True)
scan.shape

In [ ]:
data_path = "/Users/celestre/work/experimental_data/speckle_tracking/speckle_tracking/darks.h5"

darks = dip.read_image(data_path, mean=True, verbose=True)
darks.shape

In [ ]:
data_path = "/Users/celestre/work/experimental_data/speckle_tracking/speckle_tracking/flats.h5"

flats = dip.read_image(data_path, mean=True, verbose=True)
flats.shape

## 2) Apply flat-field correction

For experimental speckle data, correcting detector offsets and illumination non-uniformity is usually worth doing before computing metrics.

In [ ]:
scan = dip.preprocessing.flat_field_correction(scan, flats=flats, darks=darks, verbose=True)

## 3) Quick visual check

A fast image + histogram view is useful before computing metrics.

In [ ]:
def show_image_summary(image: np.ndarray, *, title: str | None = None,
                       bin_min: int = 0, bin_max: int = 65535, hist: bool = True, ) -> None:
    
    pmin, pmax = 1., 99.

    vmin = np.nanpercentile(image, pmin)
    vmax = np.nanpercentile(image, pmax)

    dip.plotting.plt_image(
        image,
        title=title,
        vmin=vmin,
        vmax=vmax,
        cmap="srw",
        display_origin="lower",
    )

    if hist:
        dip.plotting.plt_histogram(
            image,
            logy=True,
            cumulative=True,
            percentiles=(1, 99),
            bin_min=bin_min,
            bin_max=bin_max,
        )

    dip.plotting.show()

show_image_summary(scan[0], bin_min=100, bin_max=25000, title="first frame (flat field corrected)")

## 4) Track a representative motif between two frames

To estimate motion with template matching, we first need a **motif** to track in the image.
Here we extract it from the **first frame** and choose its size as **3× the mean speckle grain size**.
This keeps enough local texture to make the motif distinctive while remaining compact.

The match is searched in the **full next frame**.  


In [ ]:
def visualise_subroi(image, slice, npix=100, title=None, vmin=None, vmax=None):
    if vmin==None:
        vmin = np.nanpercentile(image, 0.50)
    if vmax==None:
        vmax = np.nanpercentile(image, 99.5)
    H, W = image.shape

    dip.plotting.plt_image(image, cmap = 'srw', title=title, roi=slice, display_origin='lower',
                           xmin=W//2-npix, xmax=W//2+npix,
                           ymin=H//2-npix, ymax=H//2+npix, vmin=vmin, vmax=vmax)
    dip.plotting.show()

In [ ]:
grain = dip.metrics.speckles.grain(scan[0], verbose=True)
mean_grain_px = float(np.max((grain["lx"], grain["ly"], grain["leq"])))
mean_grain_px

In [ ]:
motif_scale = 3
motif_px = dip.geometry.roi.odd_size(motif_scale * mean_grain_px)
motif_px

T, H, W = scan.shape

In [ ]:
sy, sx = dip.geometry.roi.roi_slices(
    image_shape=(H, W),
    size_yx=(motif_px, motif_px),
    center_yx=(H // 2, W // 2),
)

In [ ]:
motif = scan[0, sy, sx]
visualise_subroi(
    scan[0],
    (sy, sx),
    title=f"tracked motif ({motif_scale}× mean grain size)",
    vmin=np.nanpercentile(motif, 1),
    vmax=np.nanpercentile(motif, 99),
)


In [ ]:
reference_image_N = scan[1, :, :]

In [ ]:
%timeit dip.signal.track_translation(motif, reference_image_N, slices_yx=(sy,sx), method='template', backend='opencv')
%timeit dip.signal.track_translation(motif, reference_image_N, slices_yx=(sy,sx), method='template', backend='skimage')

In [ ]:
def offset_slice(s: slice, d: int) -> slice:
    """Shift a slice by an integer pixel offset."""
    if not isinstance(s, slice):
        raise TypeError(f"Expected slice, got {type(s)!r}")

    start = None if s.start is None else s.start + d
    stop  = None if s.stop  is None else s.stop  + d
    return slice(start, stop, s.step)

In [ ]:
dy, dx, peak, snr = dip.signal.track_translation(motif, reference_image_N, slices_yx=(sy, sx), method='template', backend='opencv')
print(f"OpenCV match: motif can be found in the search are by moving ({dy:.2f}, {dx:.2f}) pixels in the DETECTOR coordinates")
visualise_subroi(reference_image_N, (offset_slice(sy, dy), offset_slice(sx, dx)), title="matched motif in frame 1 (OpenCV)",
                 vmin=np.nanpercentile(motif, 1), vmax=np.nanpercentile(motif, 99))

In [ ]:
dy, dx, peak, snr = dip.signal.track_translation(motif, reference_image_N, slices_yx=(sy, sx), method='template', backend='skimage')
print(f"scikit-image match: motif can be found in the search are by moving ({dy:.2f}, {dx:.2f}) pixels in the DETECTOR coordinates")
visualise_subroi(reference_image_N, (offset_slice(sy, dy), offset_slice(sx, dx)), title='matched - skimage',
                 vmin=np.nanpercentile(motif, 1), vmax=np.nanpercentile(motif, 99))

## 5) Compute temporal statistics for the full stack

The stack routine tracks frame-to-frame evolution across the full scan and returns the temporal displacement series.
The plots below show the **absolute displacement** in trajectory form and as frame-indexed traces.


In [ ]:
stack_stats = dip.speckle_stack_stats(scan)

In [ ]:
dip.plotting.plt_displacement(stack_stats, kind="trajectory", temporal='abs')
dip.plotting.plt_displacement(stack_stats, kind="timeseries", temporal='abs')

dip.plotting.show()

## 6) Compare the measured displacement with the nominal scan path

For this internal diagnostic, we overlay the measured speckle displacement with the ideal spiral trajectory used during the scan.
The first plot compares the nominal and measured paths directly.  
The second plot shows the residual displacement, frame by frame.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
def spiral(x0=0, y0=0, d=25, rmax=150, initialDirection="W", clockwise=True):
    getlength = 1
    for nn in range(int(1.0 * rmax / d)):
        r = (nn + 1) * d
        u = 2 * np.pi * r
        nu = int(u / d + 0.5)
        getlength += nu
    print("points on spiral: %d " % getlength)
    xarr = np.zeros(getlength)
    yarr = np.zeros(getlength)
    xarr[0] = x0
    yarr[0] = y0
    ctr = 1
    initDir = {"E": 0, "S": np.pi / 2, "W": np.pi, "N": 3 * np.pi / 2}
    alpha = initDir[initialDirection]
    if clockwise:
        sign = 1
    else:
        sign = -1
    for nn in range(int(1.0 * rmax / d)):
        r = (nn + 1) * d
        u = 2 * np.pi * r
        nu = int(u / d + 0.5)
        for ii in range(nu):
            xarr[ctr] = x0 + r * np.sin(alpha + 2 * np.pi / nu * ii * sign)
            yarr[ctr] = y0 + r * np.cos(alpha + 2 * np.pi / nu * ii * sign)
            ctr += 1
        alpha = alpha + 2 * np.pi / nu * (nu - 1) * sign
    return xarr, yarr

In [ ]:
xarr, yarr = spiral(d=stack_stats["temporal"]["abs"]["r"][1], rmax=300)

In [ ]:
%matplotlib widget

measured_dx = stack_stats["temporal"]["abs"]["dx"]
measured_dy = stack_stats["temporal"]["abs"]["dy"]
frame_index = np.arange(measured_dx.size, dtype=float)

fig_h = 6.0
fig_w = 6.0
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

ax.plot(measured_dx, measured_dy, linewidth=1.0, color="black")
sc = ax.scatter(
    measured_dx,
    measured_dy,
    c=frame_index,
    cmap="viridis",
    s=25,
    zorder=3,
    edgecolors="black",
    linewidths=0.5,
)
ax.scatter(xarr, yarr, color="red", s=25, zorder=0, edgecolors="black", linewidths=0.5)
ax.set_xlabel("dx (px)")
ax.set_ylabel("dy (px)")
ax.set_title("Measured displacement over the nominal spiral")
ax.set_aspect(1)

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="4%", pad=0.08)
cbar = fig.colorbar(sc, cax=cax)
cbar.set_label("frame")

ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
dxp = stack_stats['temporal']['abs']['dx']
dyp = stack_stats['temporal']['abs']['dy']

fig_h = 6.0
fig_w = 6.0
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
t = np.arange(dxp.size, dtype=float)

ax.plot(xarr-dxp, yarr-dyp, linewidth=1., color='black')
sc = ax.scatter(xarr-dxp, yarr-dyp, c=t, cmap='viridis', s=25, zorder=3, edgecolors='black', linewidths=0.5)
ax.set_xlabel("dx (px)")
ax.set_ylabel("dy (px)")
ax.set_title("speckle displacement (diff)", fontsize=15)
ax.set_aspect(1)

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="4%", pad=0.08)
cbar = fig.colorbar(sc, cax=cax)
cbar.set_label("(frame)")

ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
%matplotlib inline

## Takeaway

- A motif sized from the measured speckle grain gives a practical and interpretable target for template matching.
- Backend timing with `opencv` and `scikit-image` is useful when choosing a tracker for large stacks.
- `dip.speckle_stack_stats(...)` extends this idea to the full scan and provides a compact summary of temporal displacement.
- Comparing the measured path with the nominal scan trajectory is a convenient internal diagnostic for motion consistency and tracking quality.
